# EViT-ToMe Augmentation Sweep - Priyanshu Split

This notebook contains 8 model-config pairs.
Each run cell executes all augmentations independently for that single pair.

In [ ]:
# Optional for fresh Colab runtime
!git clone https://github.com/Chalhotra/ViT-Token-Economy.git
%cd ViT-Token-Economy

In [ ]:
!git checkout priyanshu/progressive-testing
!pip -q install -r requirements.txt
!pip -q install -e .

In [ ]:
from augmentation import horizontal_flip_only, jitter_only, rotation_only, gaussian_noise_only
from src.imagenet_mapping import build_imagenet100_to_1k_map
from src.models import ModelConfig, create_model, shrink_imagenet1k_head_to_imagenet100
from src.data import DataConfig, load_imagenet100_split, build_transform_for_model, apply_timm_preprocess, build_loader
from src.eval import evaluate_accuracy_latency_throughput, evaluate_with_topk_predictions, compute_gflops
from src.utils import get_device, num_params
from src.test_models.evit_tome import EVITToMeConfig, apply_evit_tome_pruning
from pathlib import Path
from datetime import datetime
import json
import uuid
import pandas as pd

In [ ]:
device = get_device()
maps = build_imagenet100_to_1k_map()
print(f'Using device: {device}')

In [ ]:
AUGMENTATION_BUILDERS = {
    'horizontal_flip': horizontal_flip_only,
    'color_jitter': jitter_only,
    'rotation': rotation_only,
    'gaussian_noise': gaussian_noise_only,
}

if 'results' not in globals():
    results = []

In [ ]:
def run_hybrid_test(model_id, cfg, config_name, batch_size=64, aug_builder=horizontal_flip_only, aug_label='horizontal_flip'):
    print('\n' + '=' * 80)
    print(f'Testing: {config_name}')
    print(f'Model: {model_id}')
    print(f'Augmentation: {aug_label}')
    print(f'EViT locs : {cfg.evit_reduction_loc}  keep_rate: {cfg.evit_keep_rate}')
    print(f'ToMe locs : {cfg.tome_reduction_loc}  keep_rate: {cfg.tome_keep_rate}')
    print('=' * 80 + '\n')

    model = create_model(ModelConfig(model_id=model_id, pretrained=True))
    model = shrink_imagenet1k_head_to_imagenet100(model, maps.new_to_old_map, num_classes=100)
    model = apply_evit_tome_pruning(model, cfg)
    model = model.to(device).eval()

    ds = load_imagenet100_split(DataConfig(split='validation'))
    transform = build_transform_for_model(model)
    aug_pipeline = aug_builder() if callable(aug_builder) else aug_builder
    ds_t = apply_timm_preprocess(ds, transform, aug_pipeline=aug_pipeline)
    loader = build_loader(ds_t, DataConfig(batch_size=batch_size, split='validation', shuffle=False))

    metrics = evaluate_accuracy_latency_throughput(model, loader, device)
    sample = ds_t[0]['pixel_values'].unsqueeze(0).to(device)
    gflops = compute_gflops(model, sample)

    result = {
        'config_name': config_name,
        'model': model_id,
        'augmentation': aug_label,
        'params_m': num_params(model) / 1e6,
        'gflops': gflops,
        **metrics,
    }

    print(f"Results for {config_name}:")
    print(f"  Top-1 Accuracy : {metrics['acc1']:.2f}%")
    print(f"  GFLOPs         : {gflops:.3f}")
    print(f"  Latency        : {metrics['latency_ms']:.2f} ms")
    print(f"  Throughput     : {metrics['throughput']:.1f} samples/sec")
    return result

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import uuid
import pandas as pd

OUTPUT_DIR = Path.cwd() / 'lbp_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_CSV = OUTPUT_DIR / 'runs.csv'
PREDICTIONS_CSV = OUTPUT_DIR / 'predictions.csv'

RUNS_COLUMNS = [
    'run_id', 'model_id', 'augmentation', 'aug_params', 'efficiency_budget',
    'profile', 'config_key', 'acc1', 'gflops', 'latency_ms', 'params_m',
    'throughput', 'timestamp',
]
PREDICTIONS_COLUMNS = [
    'run_id', 'image_id', 'ground_truth_label', 'rank', 'predicted_class',
    'confidence', 'is_correct', 'entropy', 'conf_gap_to_rank1',
]

def _append_csv_rows(csv_path, rows, dedupe_subset):
    frame = pd.DataFrame(rows)
    if csv_path.exists():
        existing = pd.read_csv(csv_path)
        frame = pd.concat([existing, frame], ignore_index=True)
        frame = frame.drop_duplicates(subset=dedupe_subset, keep='last')
    frame.to_csv(csv_path, index=False)

def run_hybrid_test(model_id, cfg, config_name, batch_size=64, aug_builder=horizontal_flip_only, aug_label='horizontal_flip', collect_predictions=False):
    print('\n' + '=' * 80)
    print(f'Testing: {config_name}')
    print(f'Model: {model_id}')
    print(f'Augmentation: {aug_label}')
    print(f'EViT locs : {cfg.evit_reduction_loc}  keep_rate: {cfg.evit_keep_rate}')
    print(f'ToMe locs : {cfg.tome_reduction_loc}  keep_rate: {cfg.tome_keep_rate}')
    print('=' * 80 + '\n')

    model = create_model(ModelConfig(model_id=model_id, pretrained=True))
    model = shrink_imagenet1k_head_to_imagenet100(model, maps.new_to_old_map, num_classes=100)
    model = apply_evit_tome_pruning(model, cfg)
    model = model.to(device).eval()

    ds = load_imagenet100_split(DataConfig(split='validation'))
    class_names = ds.features['label'].names if hasattr(ds.features['label'], 'names') else [str(i) for i in range(100)]
    transform = build_transform_for_model(model)
    aug_pipeline = aug_builder() if callable(aug_builder) else aug_builder
    ds_t = apply_timm_preprocess(ds, transform, aug_pipeline=aug_pipeline)
    loader = build_loader(ds_t, DataConfig(batch_size=batch_size, split='validation', shuffle=False))

    if collect_predictions:
        metrics, prediction_rows = evaluate_with_topk_predictions(model, loader, device, class_names=class_names, topk=10)
    else:
        metrics = evaluate_accuracy_latency_throughput(model, loader, device)
        prediction_rows = []

    sample = ds_t[0]['pixel_values'].unsqueeze(0).to(device)
    gflops = compute_gflops(model, sample)

    result = {
        'config_name': config_name,
        'model': model_id,
        'augmentation': aug_label,
        'params_m': num_params(model) / 1e6,
        'gflops': gflops,
        **metrics,
    }

    print(f"Results for {config_name}:")
    print(f"  Top-1 Accuracy : {metrics['acc1']:.2f}%")
    print(f"  GFLOPs         : {gflops:.3f}")
    print(f"  Latency        : {metrics['latency_ms']:.2f} ms")
    print(f"  Throughput     : {metrics['throughput']:.1f} samples/sec")
    return (result, prediction_rows) if collect_predictions else result

def run_explicit_hybrid_test(model_id, config_key, reduction_loc, evit_keep_rate, tome_keep_rate, batch_size=64, aug_builder=horizontal_flip_only, aug_name='horizontal_flip'):
    cfg = EVITToMeConfig(
        evit_enabled=True,
        evit_keep_rate=tuple(evit_keep_rate),
        evit_reduction_loc=tuple(reduction_loc),
        evit_exponentiate=False,
        tome_enabled=True,
        tome_keep_rate=tuple(tome_keep_rate),
        tome_reduction_loc=tuple(reduction_loc),
        tome_exponentiate=False,
        tome_prop_attn=True,
        viz_mode=False,
    )

    run_key = f'{config_key}__{aug_name}'
    config_name = f'Hybrid {model_id} {run_key}'
    result, prediction_rows = run_hybrid_test(
        model_id=model_id,
        cfg=cfg,
        config_name=config_name,
        batch_size=batch_size,
        aug_builder=aug_builder,
        aug_label=aug_name,
        collect_predictions=True,
    )

    run_id = f'run_{model_id}__{config_key}__{aug_name}'
    timestamp = datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
    aug_params = json.dumps({
        'reduction_loc': list(reduction_loc),
        'evit_keep_rate': list(evit_keep_rate),
        'tome_keep_rate': list(tome_keep_rate),
        'batch_size': batch_size,
        'augmentation': aug_name,
    }, sort_keys=True)

    result['run_id'] = run_id
    result['model_id'] = model_id
    result['config_key'] = run_key
    result['base_config_key'] = config_key
    result['augmentation'] = aug_name
    result['efficiency_budget'] = float(config_key.split('_')[1])
    result['profile'] = '_'.join(config_key.split('_')[2:])
    result['reduction_loc'] = list(reduction_loc)
    result['evit_keep_rate'] = list(evit_keep_rate)
    result['tome_keep_rate'] = list(tome_keep_rate)
    result['aug_params'] = aug_params
    result['timestamp'] = timestamp

    run_record = {column: result.get(column) for column in RUNS_COLUMNS}
    _append_csv_rows(RUNS_CSV, [run_record], dedupe_subset=['run_id'])

    prediction_records = []
    for row in prediction_rows:
        row = dict(row)
        row['run_id'] = run_id
        prediction_records.append({column: row.get(column) for column in PREDICTIONS_COLUMNS})

    _append_csv_rows(PREDICTIONS_CSV, prediction_records, dedupe_subset=['run_id', 'image_id', 'rank'])

    print(f'Saved run summary to {RUNS_CSV}')
    print(f'Saved top-10 predictions to {PREDICTIONS_CSV}')
    return result

In [ ]:
def run_single_config(model_id, config_key, reduction_loc, evit_keep_rate, tome_keep_rate, aug_name, aug_builder, batch_size=64):
    run_key = f'{config_key}__{aug_name}'
    config_name = f'Hybrid {model_id} {run_key}'

    global results
    results = [x for x in results if x.get('config_name') != config_name]

    out = run_explicit_hybrid_test(
        model_id=model_id,
        config_key=config_key,
        reduction_loc=reduction_loc,
        evit_keep_rate=evit_keep_rate,
        tome_keep_rate=tome_keep_rate,
        batch_size=batch_size,
        aug_builder=aug_builder,
        aug_name=aug_name,
    )
    results.append(out)
    print(f'Stored result for {model_id} | {config_key} | {aug_name}')

## Priyanshu Split Runs (32 Cells)
Run cells independently. Each run cell executes one tuple: (model, augmentation, hybrid config).

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.25_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.50, 0.50, 0.50),
    tome_keep_rate=(0.50, 0.50, 0.50),
    aug_name='horizontal_flip',
    aug_builder=horizontal_flip_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.25_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.50, 0.50, 0.50),
    tome_keep_rate=(0.50, 0.50, 0.50),
    aug_name='color_jitter',
    aug_builder=jitter_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.25_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.50, 0.50, 0.50),
    tome_keep_rate=(0.50, 0.50, 0.50),
    aug_name='rotation',
    aug_builder=rotation_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.25_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.50, 0.50, 0.50),
    tome_keep_rate=(0.50, 0.50, 0.50),
    aug_name='gaussian_noise',
    aug_builder=gaussian_noise_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.25_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.50, 0.50, 0.50),
    tome_keep_rate=(0.50, 0.50, 0.50),
    aug_name='horizontal_flip',
    aug_builder=horizontal_flip_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.25_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.50, 0.50, 0.50),
    tome_keep_rate=(0.50, 0.50, 0.50),
    aug_name='color_jitter',
    aug_builder=jitter_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.25_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.50, 0.50, 0.50),
    tome_keep_rate=(0.50, 0.50, 0.50),
    aug_name='rotation',
    aug_builder=rotation_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.25_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.50, 0.50, 0.50),
    tome_keep_rate=(0.50, 0.50, 0.50),
    aug_name='gaussian_noise',
    aug_builder=gaussian_noise_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.25_pruning_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.40, 0.40, 0.40),
    tome_keep_rate=(0.625, 0.625, 0.625),
    aug_name='horizontal_flip',
    aug_builder=horizontal_flip_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.25_pruning_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.40, 0.40, 0.40),
    tome_keep_rate=(0.625, 0.625, 0.625),
    aug_name='color_jitter',
    aug_builder=jitter_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.25_pruning_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.40, 0.40, 0.40),
    tome_keep_rate=(0.625, 0.625, 0.625),
    aug_name='rotation',
    aug_builder=rotation_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.25_pruning_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.40, 0.40, 0.40),
    tome_keep_rate=(0.625, 0.625, 0.625),
    aug_name='gaussian_noise',
    aug_builder=gaussian_noise_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.25_pruning_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.40, 0.40, 0.40),
    tome_keep_rate=(0.625, 0.625, 0.625),
    aug_name='horizontal_flip',
    aug_builder=horizontal_flip_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.25_pruning_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.40, 0.40, 0.40),
    tome_keep_rate=(0.625, 0.625, 0.625),
    aug_name='color_jitter',
    aug_builder=jitter_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.25_pruning_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.40, 0.40, 0.40),
    tome_keep_rate=(0.625, 0.625, 0.625),
    aug_name='rotation',
    aug_builder=rotation_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.25_pruning_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.40, 0.40, 0.40),
    tome_keep_rate=(0.625, 0.625, 0.625),
    aug_name='gaussian_noise',
    aug_builder=gaussian_noise_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.25_merging_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.625, 0.625, 0.625),
    tome_keep_rate=(0.40, 0.40, 0.40),
    aug_name='horizontal_flip',
    aug_builder=horizontal_flip_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.25_merging_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.625, 0.625, 0.625),
    tome_keep_rate=(0.40, 0.40, 0.40),
    aug_name='color_jitter',
    aug_builder=jitter_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.25_merging_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.625, 0.625, 0.625),
    tome_keep_rate=(0.40, 0.40, 0.40),
    aug_name='rotation',
    aug_builder=rotation_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.25_merging_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.625, 0.625, 0.625),
    tome_keep_rate=(0.40, 0.40, 0.40),
    aug_name='gaussian_noise',
    aug_builder=gaussian_noise_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.25_merging_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.625, 0.625, 0.625),
    tome_keep_rate=(0.40, 0.40, 0.40),
    aug_name='horizontal_flip',
    aug_builder=horizontal_flip_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.25_merging_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.625, 0.625, 0.625),
    tome_keep_rate=(0.40, 0.40, 0.40),
    aug_name='color_jitter',
    aug_builder=jitter_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.25_merging_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.625, 0.625, 0.625),
    tome_keep_rate=(0.40, 0.40, 0.40),
    aug_name='rotation',
    aug_builder=rotation_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.25_merging_heavy',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.625, 0.625, 0.625),
    tome_keep_rate=(0.40, 0.40, 0.40),
    aug_name='gaussian_noise',
    aug_builder=gaussian_noise_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.5_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.7071, 0.7071, 0.7071),
    tome_keep_rate=(0.7071, 0.7071, 0.7071),
    aug_name='horizontal_flip',
    aug_builder=horizontal_flip_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.5_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.7071, 0.7071, 0.7071),
    tome_keep_rate=(0.7071, 0.7071, 0.7071),
    aug_name='color_jitter',
    aug_builder=jitter_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.5_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.7071, 0.7071, 0.7071),
    tome_keep_rate=(0.7071, 0.7071, 0.7071),
    aug_name='rotation',
    aug_builder=rotation_only,
)

In [ ]:
run_single_config(
    model_id='vit_tiny_patch16_224',
    config_key='eff_0.5_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.7071, 0.7071, 0.7071),
    tome_keep_rate=(0.7071, 0.7071, 0.7071),
    aug_name='gaussian_noise',
    aug_builder=gaussian_noise_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.5_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.7071, 0.7071, 0.7071),
    tome_keep_rate=(0.7071, 0.7071, 0.7071),
    aug_name='horizontal_flip',
    aug_builder=horizontal_flip_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.5_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.7071, 0.7071, 0.7071),
    tome_keep_rate=(0.7071, 0.7071, 0.7071),
    aug_name='color_jitter',
    aug_builder=jitter_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.5_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.7071, 0.7071, 0.7071),
    tome_keep_rate=(0.7071, 0.7071, 0.7071),
    aug_name='rotation',
    aug_builder=rotation_only,
)

In [ ]:
run_single_config(
    model_id='vit_base_patch16_224',
    config_key='eff_0.5_balanced',
    reduction_loc=(3, 6, 9),
    evit_keep_rate=(0.7071, 0.7071, 0.7071),
    tome_keep_rate=(0.7071, 0.7071, 0.7071),
    aug_name='gaussian_noise',
    aug_builder=gaussian_noise_only,
)

In [ ]:
print(f'Total results stored: {len(results)}')
display(pd.DataFrame(results).tail(12))